# 고유값 분해: 행렬 구조 이해 - 고유값과 고유벡터의 기하학

- Tutorial ID: `math-3-1`
- Tutorial: 고유값 분해: 행렬 구조 이해
- Section ID: `math-3-1-1`
- Section: 고유값과 고유벡터의 기하학


# 고유값과 고유벡터의 기하학 — 처음 배우는 사람을 위한 상세 노트

이 노트북은 고유값(eigenvalue)·고유벡터(eigenvector)를 **처음 배우는 사람**을 위해 만들어졌습니다.
공식을 외워서 한 번에 답을 구하는 것이 아니라, **행렬이 벡터에게 실제로 무엇을 하는지**를 숫자와 코드로 직접 확인하면서 감을 잡는 것이 목표입니다.

### 이 노트북의 구성
1. **선형변환이란?** — 행렬이 벡터를 어떻게 바꾸는지부터 출발합니다.
2. **고유벡터/고유값의 정의** — "방향이 바뀌지 않는 특별한 벡터"를 직접 찾아봅니다.
3. **기하학적 의미** — 확대/축소(스케일), 회전이 고유값에 어떻게 나타나는지 비교합니다.
4. **대각화와 행렬 거듭제곱** — 고유값/고유벡터로 계산을 쉽게 만드는 방법 + 피보나치 수열 보너스 예제.
5. **스펙트럼 분석** — 큰 행렬에서 고유값들의 분포가 신호의 증폭/감쇠와 어떻게 연결되는지 봅니다.
6. **(응용) 트랜스포머의 OV 회로 분석** — 위 개념들을 이용해 어텐션 헤드의 '복사' 동작을 탐지하는 실제 연구 사례를 살펴봅니다. (가장 어려운 부분이라 마지막에 배치했습니다.)

### 이 노트북을 읽는 법
- 위에서 아래로 순서대로, 코드 셀을 하나씩 실행하면서 출력을 직접 확인하세요.
- 숫자 자체를 외우려 하지 말고, **"방향이 바뀌는가/안 바뀌는가"**, **"크기가 커지는가/작아지는가"** 같은 패턴을 눈으로 따라가세요.
- 각 코드 셀에는 무엇을 계산하는지, 왜 그렇게 계산하는지 주석으로 설명을 달아두었습니다.
- 행렬의 숫자, seed, 차원 등을 바꿔보면서 결과가 어떻게 달라지는지 직접 실험해보면 이해가 훨씬 빨라집니다.

> 참고: 이 노트북은 numpy만 사용하므로 별도 설치 없이 바로 실행됩니다.

In [ ]:
import numpy as np

print("=" * 60)
print("고유값 분해: 행렬 구조 이해")
print("=" * 60)

## 1. 선형변환이란? — 행렬이 벡터에게 하는 일

고유벡터를 이해하려면 먼저 "행렬이 벡터에 무엇을 하는지"부터 알아야 합니다.

행렬 A를 벡터 v에 곱하면(A @ v), 새로운 벡터가 나옵니다. 이 새로운 벡터는 원래 벡터 v와 비교했을 때:
- **길이(크기)**가 달라질 수 있고
- **방향(각도)**도 달라질 수 있습니다

즉, 행렬은 "벡터를 다른 벡터로 보내는 함수"라고 생각할 수 있습니다. 이런 함수를 **선형변환(linear transformation)**이라고 부릅니다.

아래 코드에서는 같은 행렬 A로 여러 방향의 벡터를 변환해 봅니다. 대부분의 벡터는 변환 후 방향이 바뀐다는 것을 직접 눈으로 확인할 수 있습니다. 그런데 혹시, **방향이 전혀 바뀌지 않는 특별한 벡터**가 있을까요? 바로 그 질문에 대한 답이 다음 섹션의 주제인 "고유벡터"입니다.

In [ ]:
# 예시 행렬: 이 행렬이 벡터를 어떻게 바꾸는지 관찰합니다
A = np.array([[3, 1],
              [0, 2]])

print("행렬 A =")
print(A)
print()

# 여러 방향의 단위벡터(길이가 1인 벡터)들을 준비합니다
# 0°, 45°, 90°, 135° 방향의 벡터를 차례로 변환해 봅니다
angles_deg = [0, 45, 90, 135]

print(f"{'원래 각도':>10} | {'원래 벡터':^16} | {'변환된 벡터':^16} | {'변환 후 각도':>10} | {'각도 변화':>10}")
print("-" * 75)

for deg in angles_deg:
    theta = np.deg2rad(deg)
    v = np.array([np.cos(theta), np.sin(theta)])  # 길이 1인 벡터

    Av = A @ v  # 행렬 A로 벡터 v를 변환

    # np.arctan2(y, x)는 벡터(x, y)가 가리키는 방향을 각도(도 단위)로 알려주는 함수
    new_deg = np.rad2deg(np.arctan2(Av[1], Av[0]))
    angle_change = new_deg - deg

    print(f"{deg:>9}° | ({v[0]:.2f}, {v[1]:.2f})     | ({Av[0]:.2f}, {Av[1]:.2f})     | {new_deg:>9.2f}° | {angle_change:>9.2f}°")

print("\n관찰 1: 45°, 90° 방향 벡터는 변환 후 각도가 바뀌었습니다 (방향이 바뀜).")
print("관찰 2: 그런데 0°와 135° 방향 벡터는 각도 변화가 0°입니다! 방향이 전혀 바뀌지 않았습니다.")
print("        길이는 늘어났지만(0°: 1배→3배), 같은 직선 위에 그대로 있습니다.")
print("        바로 이런 방향의 벡터가 다음 섹션에서 배울 '고유벡터'입니다. 직접 확인해봅니다.")

## 2. 고유벡터와 고유값 — 방향을 바꾸지 않는 특별한 벡터

앞에서 본 것처럼 보통의 벡터는 행렬을 곱하면 방향이 바뀝니다. 그런데 **어떤 특별한 벡터는 방향은 그대로 유지하고 길이만 늘어나거나 줄어듭니다.** 이런 벡터를 **고유벡터(eigenvector)**라고 부르고, 그때 늘어나거나 줄어드는 비율을 **고유값(eigenvalue, 보통 그리스 문자 λ로 표기)**이라고 부릅니다.

수식으로 쓰면 다음과 같습니다.

**A v = λ v**

- A : 정사각 행렬 (선형변환)
- v : 고유벡터 (영벡터가 아니면서, 변환 후에도 방향이 유지되는 벡터)
- λ(람다) : 고유값 (그 방향으로 벡터가 몇 배 늘어나거나 줄어드는지를 나타내는 숫자)

이 식의 의미를 풀어보면: **"행렬 A를 v에 곱한 결과가, v에 그냥 숫자 λ를 곱한 것과 똑같다"** = **"방향은 안 변하고 크기만 λ배가 됐다"**는 뜻입니다.

numpy의 `np.linalg.eig(A)` 함수를 쓰면 한 행렬의 고유값들과 고유벡터들을 한꺼번에 계산해줍니다. 아래 코드에서는 이 정의가 실제로 맞는지 직접 검증하고, 고유벡터와 일반 벡터를 나란히 비교해 봅니다.

In [ ]:
A = np.array([[3, 1],
              [0, 2]])

# np.linalg.eig: 고유값과 고유벡터를 동시에 계산
# eigenvalues: 고유값들 (1차원 배열)
# eigenvectors: 고유벡터들 (각 "열"이 하나의 고유벡터에 해당함, 길이 1로 정규화되어 있음)
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"A = \n{A}\n")
print(f"고유값: {eigenvalues}")
print(f"고유벡터 (열 벡터로 저장됨):\n{eigenvectors}\n")

# === 검증 1: 고유벡터는 정의(A @ v = λ * v)를 만족하는가? ===
print("[검증] A @ v 와 λ * v 가 같은지 확인")
print("-" * 50)
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]      # i번째 고유벡터 (열 하나를 꺼냄)
    lam = eigenvalues[i]        # 그에 대응하는 고유값

    Av = A @ v        # 행렬을 곱해서 변환
    lam_v = lam * v   # 단순히 숫자 λ만 곱함

    print(f"\n{i+1}번째 고유벡터 v = {np.round(v, 4)}, 고유값 λ = {lam:.4f}")
    print(f"  A @ v   = {np.round(Av, 4)}")
    print(f"  λ * v   = {np.round(lam_v, 4)}")
    print(f"  두 결과가 같은가? {np.allclose(Av, lam_v)}  ← 같다면 고유벡터가 맞다는 뜻")

# === 검증 2: 고유벡터는 방향이 안 바뀌고, 일반 벡터는 방향이 바뀐다 ===
print("\n\n[비교] 고유벡터 vs 일반 벡터 — 변환 후 방향이 바뀌는가?")
print("-" * 50)

def angle_deg(vec):
    """벡터의 방향(각도, 도 단위)을 계산하는 함수"""
    return np.rad2deg(np.arctan2(vec[1], vec[0]))

# 고유벡터 하나를 가져와서 변환 전후 각도를 비교
v_eigen = eigenvectors[:, 0]
Av_eigen = A @ v_eigen
print(f"고유벡터        : 변환 전 각도 {angle_deg(v_eigen):7.2f}°  →  변환 후 각도 {angle_deg(Av_eigen):7.2f}°  (같은 직선 위, 방향 유지)")

# 임의로 고른 일반 벡터 (고유벡터가 아닌 벡터)
v_random = np.array([1.0, 1.0])
Av_random = A @ v_random
print(f"일반 벡터(1,1)  : 변환 전 각도 {angle_deg(v_random):7.2f}°  →  변환 후 각도 {angle_deg(Av_random):7.2f}°  (방향이 바뀜)")

## 3. 기하학적 의미 — 고유값이 알려주는 것

고유값은 "그 고유벡터 방향으로 벡터가 얼마나 늘어나거나 줄어드는지"를 알려줍니다. 몇 가지 대표적인 변환을 통해 고유값의 의미를 직관적으로 살펴봅니다.

- 고유값이 **1보다 크면** → 그 방향으로 벡터가 늘어남 (확대)
- 고유값이 **0과 1 사이면** → 그 방향으로 벡터가 줄어듦 (축소)
- 고유값이 **음수면** → 그 방향이 반대로 뒤집힘 (방향 반전 + 크기 변화)
- 고유값이 **복소수(complex number)면** → 실수 고유벡터가 없다는 뜻. 보통 변환에 "회전"이 섞여 있다는 신호입니다

먼저 가장 단순한 경우인 **스케일(확대/축소) 행렬**부터 봅니다.

In [ ]:
# 이 행렬은 x축 방향은 2배로 늘리고, y축 방향은 절반으로 줄입니다.
# 대각행렬(diagonal matrix, 대각선 외의 값이 모두 0인 행렬)이라서
# 고유벡터가 정확히 x축, y축 방향이 됩니다.
S = np.array([[2, 0],
              [0, 0.5]])

evals, evecs = np.linalg.eig(S)
print("스케일 행렬 S =")
print(S)
print(f"고유값: {evals}")
print(f"고유벡터:\n{np.round(evecs, 4)}")
print("해석:")
print(f"  고유값 {evals[0]} → 이 방향(x축) 벡터는 길이가 {evals[0]}배가 됨 (확대)")
print(f"  고유값 {evals[1]} → 이 방향(y축) 벡터는 길이가 {evals[1]}배가 됨 (축소)")

# 실제로 확인: x축 방향 단위벡터 (1,0)을 S에 통과시키면?
v_x = np.array([1.0, 0.0])
print(f"\n  예) (1,0)을 S에 통과시키면 → {S @ v_x}  (x좌표만 2배)")
v_y = np.array([0.0, 1.0])
print(f"      (0,1)을 S에 통과시키면 → {S @ v_y}  (y좌표만 0.5배)")

### 회전 행렬은 왜 다를까?

스케일 행렬은 x축, y축 방향이 그대로 유지되면서 길이만 바뀌었습니다. 그런데 **회전 행렬**은 어떨까요? 회전은 "모든 벡터를 똑같은 각도만큼 돌리는" 변환입니다. 만약 모든 벡터의 방향이 똑같이 돌아간다면, **변환 후에도 원래와 같은 방향을 유지하는 벡터가 하나라도 있을까요?** (0°나 180° 회전이 아닌 경우를 생각해보세요.) 직접 코드로 확인해 봅니다.

In [ ]:
print("회전 행렬: 모든 벡터의 방향을 똑같이 돌리면 어떻게 될까?")
print("-" * 50)

theta = np.pi / 4  # 45도 = pi/4 라디안
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print(f"45도 회전 행렬 R =\n{np.round(R, 4)}\n")

# 먼저, 정말로 "모든" 벡터의 방향이 바뀌는지 여러 각도로 확인해봅니다
print("여러 방향의 벡터를 45도 회전 행렬에 통과시켜 봅니다:")
for deg in [0, 30, 90, 160]:
    th = np.deg2rad(deg)
    v = np.array([np.cos(th), np.sin(th)])
    Rv = R @ v
    new_deg = np.rad2deg(np.arctan2(Rv[1], Rv[0]))
    print(f"  {deg:>3}° 방향 벡터 → 변환 후 {new_deg:>6.2f}° 방향  (항상 +45도 만큼 돌아감)")

print("\n→ 어떤 방향에서 시작하든 항상 45도만큼 돌아갑니다.")
print("  즉, '변환 후에도 원래 방향과 같은 직선 위에 있는' 실수 벡터가 하나도 없습니다.")
print("  이것이 바로 회전 행렬에 '실수 고유벡터가 없는' 이유입니다.\n")

evals_r, evecs_r = np.linalg.eig(R)
print(f"실제로 numpy로 고유값을 계산해보면: {np.round(evals_r, 4)}")
print("→ 고유값이 복소수(j가 포함된 수)로 나옵니다. 실수 고유벡터가 없다는 신호입니다.")
print(f"\n고유값의 절댓값(크기) |λ| = {np.abs(evals_r[0]):.4f}")
print("→ 크기가 정확히 1입니다. 즉 회전은 벡터의 '길이'는 바꾸지 않고 '방향'만 바꾼다는 뜻입니다.")
print(f"  실제로 고유값의 각도: {np.rad2deg(np.angle(evals_r[0])):.1f}°, {np.rad2deg(np.angle(evals_r[1])):.1f}°  ← 회전각 ±45°와 정확히 일치!")
print("  (복소 고유값의 '각도' 부분이 바로 회전 각도와 대응됩니다)")

## 4. 대각화 — 고유값/고유벡터로 계산을 쉽게 만들기

고유값·고유벡터를 알면 좋은 점 하나: **행렬을 여러 번 곱하는 계산(거듭제곱, A^n)이 훨씬 쉬워집니다.**

각 고유벡터 vᵢ에 대해 A vᵢ = λᵢ vᵢ가 성립합니다. 모든 고유벡터를 모아 행렬 V의 열로 쌓고, 고유값들을 대각선에 놓은 행렬을 D라고 하면, 이 관계를 한꺼번에 다음과 같이 쓸 수 있습니다.

**A V = V D  ⟹  A = V D V⁻¹**

(고유벡터들이 서로 독립이라서 V가 역행렬을 가질 때 가능합니다.) 이것을 **대각화(diagonalization)**라고 부릅니다.

이게 왜 유용할까요? A를 n번 곱하면:

**A^n = (V D V⁻¹)(V D V⁻¹) ⋯ (V D V⁻¹) = V D^n V⁻¹**

중간의 V⁻¹V가 매번 사라지고 D만 남기 때문입니다. 그리고 D는 대각행렬이라 D^n은 그냥 **대각 성분(고유값)을 각각 n번 거듭제곱**하면 됩니다 — 행렬을 n번 곱하는 대신, 숫자를 n번 거듭제곱하는 훨씬 쉬운 문제로 바뀌는 것입니다!

아래에서 이 과정을 직접 확인하고, 마지막에는 이 원리를 실제로 활용하는 유명한 예시(피보나치 수열)도 살펴봅니다.

In [ ]:
print("4. 대각화: A = V D V⁻¹")
print("-" * 45)

A = np.array([[4, 1],
              [2, 3]])

evals, V = np.linalg.eig(A)
D = np.diag(evals)          # 고유값들을 대각선에 놓은 행렬
V_inv = np.linalg.inv(V)    # V의 역행렬

print(f"A = \n{A}")
print(f"\n고유값 D = diag({np.round(evals, 4)})")
print(f"고유벡터 V = \n{np.round(V, 4)}")

# 검증: A = V @ D @ V^(-1) 인지 확인
A_reconstructed = V @ D @ V_inv
print(f"\nV @ D @ V⁻¹ = \n{np.round(np.real(A_reconstructed), 4)}")
print(f"원래 A와 같은가? {np.allclose(A, A_reconstructed)}")

# 행렬 거듭제곱이 쉬워짐!
print("\n행렬 거듭제곱의 효율적 계산:")
print("  A^n = V @ D^n @ V⁻¹  (D^n은 대각 성분을 각각 n번 거듭제곱하면 끝)")
n = 10
A_power_direct = np.linalg.matrix_power(A, n)   # 방법 1: 그냥 n번 직접 곱하기
D_power = np.diag(evals ** n)                    # 방법 2: 고유값만 n번 거듭제곱
A_power_eigen = V @ D_power @ V_inv

print(f"\nA^{n} (직접 계산):\n{np.round(np.real(A_power_direct), 1)}")
print(f"A^{n} (고유값 사용):\n{np.round(np.real(A_power_eigen), 1)}")
print(f"같은가? {np.allclose(A_power_direct, A_power_eigen)}")

### 보너스: 대각화로 피보나치 수열 빠르게 계산하기

대각화가 실제로 쓸모 있다는 걸 보여주는 유명한 예시가 **피보나치 수열**입니다. 피보나치 수열은 점화식 F(n) = F(n-1) + F(n-2)로 정의되는데, 이 점화식을 행렬로 표현하면 **행렬 거듭제곱 문제**가 됩니다. 즉, 바로 위에서 배운 대각화를 그대로 활용할 수 있습니다.

In [ ]:
print("피보나치 수열: 1, 1, 2, 3, 5, 8, 13, 21, ... (앞 두 수를 더해서 다음 수를 만듦)")
print("F(n) = F(n-1) + F(n-2),  F(0)=0, F(1)=1\n")

print("이 점화식을 행렬로 표현할 수 있습니다:")
print("  [F(n+1)]   [1 1] [F(n)  ]")
print("  [F(n)  ] = [1 0] [F(n-1)]")
print()
print("즉 M = [[1,1],[1,0]] 을 n번 곱하면 한 번에 F(n)을 구할 수 있습니다.\n")

M = np.array([[1, 1],
              [1, 0]])

evals_M, V_M = np.linalg.eig(M)
print(f"M의 고유값: {np.round(evals_M, 4)}")
print("→ 이 중 큰 값(약 1.618...)이 바로 유명한 '황금비(golden ratio, φ)'입니다!\n")

# M^n을 고유값 분해로 빠르게 계산
n = 15
D_M = np.diag(evals_M ** n)
M_n = np.real(V_M @ D_M @ np.linalg.inv(V_M))

# [F(1), F(0)] = [1, 0] 에서 시작해서 M^n을 곱하면 [F(n+1), F(n)]이 나옴
start = np.array([1, 0])
result = M_n @ start
print(f"M^{n} 을 이용해 구한 피보나치 수: F({n}) ≈ {result[1]:.4f}")

# 반복문으로 직접 계산한 값과 비교 (검증용)
fib = [0, 1]
for i in range(2, n + 1):
    fib.append(fib[-1] + fib[-2])
print(f"반복문으로 직접 구한 값      : F({n}) = {fib[n]}")
print(f"두 값이 거의 같은가? {np.isclose(result[1], fib[n])}")

## 5. 스펙트럼 분석 — 큰 행렬에서 신호는 어떻게 변할까?

지금까지는 2×2처럼 작은 행렬을 다뤘습니다. 그런데 신경망에서는 수백~수천 차원의 큰 행렬을 다룹니다. 큰 행렬의 고유값을 하나하나 손으로 확인하긴 어렵지만, **"고유값들이 대체로 어떤 범위에 분포하는지"**는 그 행렬이 신호(signal)를 어떻게 다루는지 알려주는 중요한 정보입니다.

두 가지 핵심 개념을 구분해서 알아둡니다.

| 개념 | 정의 | 알려주는 것 |
|---|---|---|
| **스펙트럼 반경**(spectral radius) | 모든 고유값 중 절댓값이 가장 큰 값, max\|λ\| | 같은 변환을 **반복(여러 번)** 적용했을 때 장기적으로 폭발하는지 사라지는지 |
| **스펙트럼 노름**(spectral norm, \|\|M\|\|₂) | 가장 큰 특이값(singular value) | **한 번** 변환을 적용했을 때 가능한 최대 확대 비율 |

이 둘은 행렬이 대칭(symmetric)이 아니면 보통 서로 다른 값입니다. 아래 코드에서는 같은 랜덤 행렬을 세 가지 강도(기본/증폭/감쇠)로 스케일링해서, 스펙트럼 반경이 신호의 장기적인 증폭·감쇠와 실제로 어떻게 연결되는지 직접 확인합니다.

In [ ]:
d = 100
np.random.seed(0)

# 기본 랜덤 행렬: 각 원소가 표준정규분포(N(0,1))를 따르고, 1/sqrt(d)로 정규화
# (이렇게 정규화하면 행렬 크기(d)가 커져도 스펙트럼 반경이 대략 1 근처에 머무릅니다.
#  이는 신경망 가중치를 초기화할 때 신호가 폭발하거나 사라지지 않도록 하는 데 쓰이는 원리와 같습니다.)
M_base = np.random.randn(d, d) / np.sqrt(d)

# 비교를 위해 같은 행렬을 살짝 다르게 스케일링한 두 버전을 추가로 만듭니다
M_amplify = M_base * 1.5   # 전체적으로 1.5배 키운 버전 → 증폭이 일어날 것으로 예상
M_decay = M_base * 0.6     # 전체적으로 0.6배 줄인 버전 → 감쇠가 일어날 것으로 예상

def report(name, M):
    evals_M = np.linalg.eigvals(M)
    spectral_radius = np.max(np.abs(evals_M))   # 모든 고유값의 절댓값 중 최댓값
    spectral_norm = np.linalg.norm(M, 2)        # 가장 큰 특이값 (한 번 곱했을 때 최대 확대 비율)
    print(f"[{name}]")
    print(f"  스펙트럼 반경 (max|λ|)  = {spectral_radius:.4f}")
    print(f"  스펙트럼 노름 (||M||_2) = {spectral_norm:.4f}")
    return spectral_radius

print(f"{d}×{d} 랜덤 행렬 세 가지를 비교합니다 (기본 / 1.5배 증폭 / 0.6배 감쇠)\n")
r_base = report("기본 (M_base)", M_base)
print()
r_amp = report("증폭 (M_amplify = 1.5 × M_base)", M_amplify)
print()
r_dec = report("감쇠 (M_decay = 0.6 × M_base)", M_decay)

# 실제로 같은 벡터를 각 행렬에 반복해서 곱하면서 길이(norm) 변화를 관찰
print("\n같은 시작 벡터를 각 행렬에 반복해서 곱했을 때 벡터 길이 변화:")
np.random.seed(1)
v0 = np.random.randn(d)
v0 = v0 / np.linalg.norm(v0)  # 길이 1로 시작

print(f"\n{'반복 횟수':>8} | {'기본':>12} | {'증폭(×1.5)':>12} | {'감쇠(×0.6)':>12}")
print("-" * 55)
for n in [1, 5, 10, 20, 30]:
    len_base = np.linalg.norm(np.linalg.matrix_power(M_base, n) @ v0)
    len_amp = np.linalg.norm(np.linalg.matrix_power(M_amplify, n) @ v0)
    len_dec = np.linalg.norm(np.linalg.matrix_power(M_decay, n) @ v0)
    print(f"{n:>8} | {len_base:>12.4e} | {len_amp:>12.4e} | {len_dec:>12.4e}")

print("\n해석:")
print(f"  · 기본 행렬: 스펙트럼 반경 ≈ {r_base:.2f} (1에 가까움) → 반복해도 길이가 폭발/소멸하지 않고 비교적 안정적")
print(f"  · 증폭 행렬: 스펙트럼 반경 ≈ {r_amp:.2f} (1보다 큼)   → 반복할수록 길이가 점점 커짐 (증폭/exploding)")
print(f"  · 감쇠 행렬: 스펙트럼 반경 ≈ {r_dec:.2f} (1보다 작음) → 반복할수록 길이가 점점 작아짐 (소멸/vanishing)")

print("\n딥러닝/트랜스포머에서의 의미:")
print("  - 같은 연산(레이어)을 깊게 반복하는 구조에서는 가중치 행렬의 스펙트럼 반경이")
print("    1보다 크면 신호가 폭발하고, 1보다 작으면 신호가 사라집니다.")
print("  - 그래서 가중치 초기화 기법들(Xavier/Glorot 초기화 등)은 스펙트럼 반경을 1 근처로 맞추려고 합니다.")
print("  - 복소 고유값이 있다면, 단순 확대/축소가 아니라 회전·진동하는 성분도 섞여 있다는 뜻입니다.")

## 6. (응용) 트랜스포머의 OV 회로 분석 — 고유값으로 '복사' 동작 찾기

지금부터는 **응용 예제**입니다. 트랜스포머(Transformer) 모델의 어텐션(attention) 메커니즘 안에는 **OV 회로(OV circuit)**라는 구성요소가 있는데, 이 부분의 고유값을 분석하면 "이 어텐션 헤드가 복사(copying) 동작을 하는가"를 수학적으로 판단할 수 있습니다. (이 아이디어는 Anthropic의 트랜스포머 회로 해석 연구에서 제안된 방법입니다.)

먼저 용어를 하나씩 풀어봅니다.

- **토큰(token)**: 모델이 다루는 단어/글자 조각. 여기서는 설명을 단순화하기 위해 어휘(vocabulary) 크기를 4로, 즉 토큰이 4종류만 있다고 가정합니다.
- **임베딩 행렬 W_E**: 각 토큰(4개 중 하나)을 d_model 차원의 벡터로 바꿔주는 행렬. "토큰 번호 → 벡터" 변환입니다.
- **언임베딩 행렬 W_U**: 모델 내부의 d_model 차원 벡터를 다시 "이 벡터가 어떤 토큰에 가까운지" 나타내는 점수(logit)로 바꿔주는 행렬. "벡터 → 토큰별 점수" 변환으로, W_E의 반대 방향 역할을 합니다.
- **Value 행렬 W_V, Output 행렬 W_O**: 어텐션이 "어떤 토큰에 주목할지" 정하고 나면, 그 토큰의 정보를 W_V로 한 번 변환하고 W_O로 다시 변환해서 다음 단계로 전달합니다. 이 둘을 합친 W_V @ W_O = W_OV를 **'OV 회로'**라고 부릅니다.

이제 네 행렬을 순서대로 합치면(W_E @ W_OV @ W_U), **'어휘 × 어휘'** 크기의 행렬 하나가 나옵니다. 이 행렬의 (i, j) 성분은 **"토큰 i에 주목했을 때, 토큰 j의 점수가 얼마나 올라가는가"**를 의미합니다.

여기서 핵심 질문 하나: 이 행렬의 **대각 성분**(즉 i=j인 경우, "토큰 i에 주목했을 때 토큰 i 자신의 점수가 올라가는가")이 양수라면 어떤 의미일까요? → "특정 토큰을 보면, 그 토큰 자신을 다시 예측하려는 경향"이 생긴다는 뜻이고, 이것이 바로 **'복사(copying)' 동작의 수학적 정의**입니다.

그리고 고유값과의 연결: 이 행렬의 고유값들이 대부분 **양의 실수**라면, 전체적으로 "자기 자신을 강화하는" 성분이 강하다고 볼 수 있습니다. (참고: 모든 고유값의 합은 대각 성분의 합과 같다는 성질(trace) 때문에, 대각 성분들이 양수일수록 양의 고유값이 나오기 쉬워집니다.) 그래서 **양의 실수 고유값의 비율**을 복사 동작을 판별하는 지표로 사용합니다.

아래 코드에서는 두 가지 OV 회로를 비교합니다.
1. **랜덤 OV 회로** — 복사와 무관하게 임의로 학습되었다고 가정한 경우
2. **복사형 OV 회로** — 단위행렬(항등행렬, identity matrix: 입력을 그대로 출력하는 행렬)에 가까운, 정보를 거의 그대로 전달하는 경우

In [ ]:
print("OV 회로의 복사 동작 감지")
print("-" * 45)

vocab_size = 4   # 어휘 크기: 토큰이 4종류 있다고 가정 (설명을 단순화하기 위한 작은 숫자)
d_model = 6      # 모델 내부에서 벡터를 표현하는 차원
d_head = 3       # 어텐션 헤드 하나가 사용하는 차원 (d_model보다 작은 게 보통)

np.random.seed(24)  # 항상 같은 난수가 나오게 고정 (재현 가능하게)

# --- 경우 1: 평범한(랜덤) OV 회로 -------------------------------------------
# 학습이 안 된, 혹은 복사와 무관한 어텐션 헤드라고 가정. W_V, W_O를 랜덤하게 생성.
W_V_random = np.random.randn(d_model, d_head) * 0.3
W_O_random = np.random.randn(d_head, d_model) * 0.3
W_OV_random = W_V_random @ W_O_random  # d_model × d_model 크기. 이게 'OV 회로'

# --- 경우 2: 복사형 OV 회로 -------------------------------------------------
# np.eye(n, m)은 '단위행렬(identity matrix)'을 만듭니다.
# 단위행렬은 어떤 벡터에 곱해도 그 벡터를 그대로 돌려주는 행렬입니다 (정보를 그대로 전달 = 복사).
# 여기에 0.8을 곱한 건 "거의 그대로 전달하되 약간만 줄여서 전달"하는 상황을 흉내낸 것입니다.
W_V_copy = np.eye(d_model, d_head) * 0.8
W_O_copy = np.eye(d_head, d_model) * 0.8
W_OV_copy = W_V_copy @ W_O_copy

# --- 임베딩 / 언임베딩 행렬 --------------------------------------------------
# W_E: 토큰(vocab_size개) → 내부 벡터(d_model차원)로 바꾸는 행렬
# W_U: 내부 벡터(d_model차원) → 토큰별 점수(vocab_size개)로 바꾸는 행렬
W_E = np.random.randn(vocab_size, d_model) * 0.4
W_U = np.random.randn(d_model, vocab_size) * 0.3

# --- 확장 OV 회로 (extended OV circuit) -------------------------------------
# W_E @ W_OV @ W_U 를 계산하면 'vocab_size × vocab_size' 행렬이 나옵니다.
# 이 행렬의 (i, j) 성분 = "토큰 i를 봤을 때, 토큰 j의 점수가 얼마나 변하는가"
C_OV_random = W_E @ W_OV_random @ W_U
C_OV_copy = W_E @ W_OV_copy @ W_U

print(f"확장 OV 회로의 크기: {C_OV_random.shape}  (vocab_size × vocab_size = {vocab_size} × {vocab_size})\n")

# --- 고유값 계산 -------------------------------------------------------------
evals_random = np.linalg.eigvals(C_OV_random)
evals_copy = np.linalg.eigvals(C_OV_copy)

# --- 판정 함수: 양의 실수 고유값이 전체 중 몇 %인지 계산 ----------------------
def positive_eigenvalue_ratio(evals):
    """
    고유값들 중 '실수부가 0.01보다 큰' 것의 비율을 계산합니다.
    (복소수 고유값도 실수부(real part)를 기준으로 판단합니다)
    비율이 높을수록 '자기 자신을 강화하는(=복사하는)' 성분이 많다는 뜻입니다.
    """
    real_parts = np.real(evals)
    return np.sum(real_parts > 0.01) / len(evals)

ratio_random = positive_eigenvalue_ratio(evals_random)
ratio_copy = positive_eigenvalue_ratio(evals_copy)

print("[경우 1] 랜덤 OV 회로:")
print(f"  고유값 (실수부, 큰 순서): {np.round(np.sort(np.real(evals_random))[::-1], 3)}")
print(f"  양의 고유값 비율: {ratio_random:.2f}")
print(f"  판정: {'복사형' if ratio_random > 0.6 else '비복사형'}  (기준: 비율 > 0.6 이면 복사형으로 판정)")

print("\n[경우 2] 복사형(항등행렬 기반) OV 회로:")
print(f"  고유값 (실수부, 큰 순서): {np.round(np.sort(np.real(evals_copy))[::-1], 3)}")
print(f"  양의 고유값 비율: {ratio_copy:.2f}")
print(f"  판정: {'복사형' if ratio_copy > 0.6 else '비복사형'}")

print("\n정리:")
print("  · 양의 실수 고유값 = 그 방향(토큰)이 자기 자신의 점수를 스스로 높이는 성분")
print("  · 항등행렬에 가까운 OV 회로(경우 2)는 '정보를 그대로 전달'하므로 양의 고유값 비율이 훨씬 높게 나옵니다")
print("  · 실제 트랜스포머를 학습시킨 뒤 이런 식으로 각 어텐션 헤드의 OV 회로를 분석하면,")
print("    어떤 헤드가 '복사 헤드(copying head, 예: induction head)' 역할을 하는지 찾아낼 수 있습니다")

## 정리 및 다음으로 해볼 것

이 노트북에서 살펴본 핵심 아이디어를 한 줄씩 정리하면:

- 행렬은 벡터를 다른 벡터로 보내는 "변환"이고, 대부분의 벡터는 변환 후 방향이 바뀝니다.
- **고유벡터**는 변환 후에도 방향이 바뀌지 않는 특별한 벡터이고, **고유값**은 그 방향에서의 확대/축소 비율입니다.
- 고유값의 부호·크기·실수/복소수 여부는 변환이 확대/축소/반전/회전 중 무엇을 하는지 알려줍니다.
- **대각화**(A = V D V⁻¹)를 쓰면 행렬의 거듭제곱을 매우 효율적으로 계산할 수 있습니다.
- 큰 행렬에서는 고유값들의 분포(**스펙트럼 반경**)가 반복된 변환에서 신호가 폭발하는지 사라지는지를 결정합니다.
- 트랜스포머 어텐션의 OV 회로처럼, 실제 모델 분석에서도 고유값은 "이 구성요소가 어떤 동작을 하는가"를 알아내는 도구로 쓰입니다.

### 직접 실험해보면 좋은 것들
- Section 2의 행렬 `A`의 값을 바꿔서, 고유값/고유벡터가 어떻게 달라지는지 확인해보세요.
- Section 3의 회전 각도(`theta`)를 0°, 90°, 180°로 바꿔보면서 언제 실수 고유벡터가 존재하는지/존재하지 않는지 확인해보세요.
- Section 4의 거듭제곱 횟수 `n`을 더 크게 바꿔서 결과를 비교해보세요.
- Section 5의 스케일링 비율(1.5, 0.6)을 바꿔보거나, 차원 `d`를 키워보면서 스펙트럼 반경이 어떻게 변하는지 관찰해보세요.
- Section 6의 `d_model`, `d_head`, `vocab_size`나 `np.random.seed()` 값을 바꿔서 '복사형' 판정 결과가 달라지는지 확인해보세요.